# Orchestrator fine-tuning · Kev (LoRA + pointer head) on Kaggle

Plan: `orchestrator/fine-tuning/plans/FT-P2-kev-0.8b.md`. Stages stop at the first failure:
`c0` (read released meta, checks) → `c1` smoke → `c2` pilot (time projection) → `c3` full run + calibration → `c4_validation` (sanity check, report) → `c4_test` (**only when `FROZEN = True`**, once).

**Before you run**
- Accelerator: **one GPU T4** (not P100). Internet: **On**. Add Input: the private Dataset built by `fine-tuning/kev/make_bundle_ft.py`.
- Optional secret `HF_TOKEN`. Zero money: free tier only (FT-C-08). Never run two training sessions on one account.
- Run `c0` to `c2` first, read the projection, then run `c3` and `c4_validation`. Set `FROZEN = True` by hand only after validation is reviewed (FT-C-14).
- Checkpoint (adapter + `head.pt`) is saved in `/kaggle/working/runs/kev/<RUN_ID>/checkpoint`: download it, upload as a private Kaggle Dataset, record the sha256 values printed at the end.

In [ ]:
# ---- CONFIG (edit here) ----
SIZE = "0.8b"                       # "0.8b" first; "4b" only after 0.8b has finished c0 and c1 (FT-P-02)
STAGES = ["c0", "c1"]               # add "c2", then "c3", "c4_validation"; "c4_test" needs FROZEN = True
FROZEN = False                      # set True by hand after validation is reviewed (FT-C-14): test is run once
KEV_REPO = "https://github.com/jaredpalmer/kev.git"
KEV_SHA = "2074cc44256200bb37540103fa4572901faa10f3"
FLA_VERSION = "0.5.2"
BOX_HOURS = {"0.8b": 2, "4b": 3}[SIZE]                      # FT-C-13: pilot + full run + calibration
SANITY_MACRO_F1 = {"0.8b": 0.357, "4b": 0.483}[SIZE]        # FT-C-12: own zero-shot development route macro-F1
PIN = {"0.8b": "9a45d25eb2ab761841196625383fa1dff0e56c1e", "4b": "139fdd94"}[SIZE]
DTYPE = {"0.8b": "fp32", "4b": "bf16"}[SIZE]
RUN_ID = f"kev-{SIZE}-ft-a"                                 # FT-P-03; a second configuration takes suffix -b

import os
OUT = "/kaggle/working/runs/kev"
WORK = "/tmp/work"
os.makedirs(WORK, exist_ok=True)
os.environ["HF_HOME"] = "/tmp/hf"
print(RUN_ID, STAGES, "FROZEN =", FROZEN)

In [ ]:
import subprocess, sys, shlex

def sh(cmd, cwd=None, check=True, env=None):
    """Run a command, stream its output into the cell, raise on failure."""
    print("$", cmd if isinstance(cmd, str) else shlex.join(cmd), flush=True)
    p = subprocess.Popen(cmd, shell=isinstance(cmd, str), cwd=cwd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, env={**os.environ, **(env or {})})
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed with exit code {p.returncode}")
    return p.returncode

sh("nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv")
sh("python --version; df -h /tmp /kaggle/working | cat; free -g | head -2")

In [ ]:
# ---- 1. code: project bundle (attached Dataset) + upstream Kev at the pinned commit ----
import glob
try:
    from kaggle_secrets import UserSecretsClient
    try: os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception: pass
except ImportError:
    pass   # not on Kaggle

hits = glob.glob("/kaggle/input/**/orchestrator/fine-tuning/kev/train_kev.py", recursive=True)
assert len(hits) == 1, f"attach exactly one Dataset built by kev/make_bundle_ft.py; found {hits}"
FT = os.path.dirname(os.path.dirname(hits[0]))             # .../orchestrator/fine-tuning (read-only)
BASE = os.path.join(os.path.dirname(FT), "baselines")
assert os.path.isfile(f"{BASE}/common/data.py") and os.path.isfile(f"{FT}/data/kev/train.jsonl") and os.path.isfile(f"{FT}/data/manifest.json")
print("bundle:", FT)

KEV_DIR = f"{WORK}/kev-src"
if not os.path.exists(KEV_DIR):
    sh(f"git clone {KEV_REPO} {KEV_DIR}")
sh(f"git checkout -q {KEV_SHA} && git rev-parse HEAD", cwd=KEV_DIR)

In [ ]:
# ---- 2. environment: own venv (Python 3.12, kev's locked deps) ----
sh("pip -q install uv")
sh("uv sync --extra serve --no-dev", cwd=KEV_DIR, env={"UV_PROJECT_ENVIRONMENT": f"{WORK}/venv"})
PY = f"{WORK}/venv/bin/python"
sh(f"uv pip install --python {PY} flash-linear-attention=={FLA_VERSION}")
sh(f"uv pip install --python {PY} -r {BASE}/requirements-common.txt")
sh([PY, "-c", "import torch, transformers, fla; print('torch', torch.__version__, '| transformers', transformers.__version__, '| fla', fla.__version__)"])

In [ ]:
# ---- 3. GPU sanity + shared tests ----
sh([PY, "-c", """
import torch
assert torch.cuda.is_available(), "no CUDA GPU: set Accelerator to a T4"
cap = torch.cuda.get_device_capability(0); arch = f"sm_{cap[0]}{cap[1]}"
print("device", torch.cuda.get_device_name(0), arch, "| bf16 supported:", torch.cuda.is_bf16_supported(), "| total GB", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1))
assert arch in torch.cuda.get_arch_list(), f"no kernels for {arch}: choose a T4"
"""])
sh(f"{PY} -m pytest -q -p no:cacheprovider common/tests kev/tests", cwd=FT, env={"PYTHONPATH": f"{KEV_DIR}:{os.path.dirname(os.path.dirname(FT))}", "KEV_SRC": KEV_DIR})
sh(f"{PY} -c \"import sys; sys.path.append('common'); import checks; checks.check_manifest('data', prefix='kev/'); print('data manifest ok (kev files)')\"", cwd=FT)
# Kaggle quota: read the current session length and weekly GPU limit from the account page and write them into the notes (chapter 09 item 6).

In [ ]:
# ---- c0: released checkpoint meta, command (dry run), upstream facts of chapter 09 ----
import json, pathlib
RUN = f"{OUT}/{RUN_ID}"
if "c0" in STAGES:
    sh(f"{PY} kev/train_kev.py --size {SIZE} --data-dir data/kev --out {RUN} --dry-run", cwd=FT)
    sh(f"grep -n 'z / self.temperature' {KEV_DIR}/kev/model.py; grep -n 'def augment\\|augment(req' {KEV_DIR}/kev/data.py {KEV_DIR}/kev/train.py; grep -n 'COMPAT_FIELDS =' {KEV_DIR}/kev/checkpoint.py")
    print("C0 done: copy the printed meta, command and grep results into run.json notes (chapter 09 items 1 and 3).")

In [ ]:
# ---- c1: smoke. 32 records, one optimizer step, reload the checkpoint with the round-1 runner ----
if "c1" in STAGES:
    SM = f"{OUT}/{RUN_ID}-c1"
    sh(f"{PY} kev/train_kev.py --size {SIZE} --data-dir data/kev --out {SM} --limit 32 --max-steps 1", cwd=FT)
    rep = json.load(open(f"{SM}/train_report.json"))
    assert rep["return_code"] == 0 and rep["dropped_message"] is None
    chk = rep["checkpoint_check"]
    assert not chk["architecture_differences"], chk
    print("base_revision", chk["base_revision"], "| released", rep["released_meta"]["base_revision"])
    assert chk["base_revision"] == rep["released_meta"]["base_revision"]
    sh(f"{PY} kev/run_ft.py --checkpoint {SM}/checkpoint --size {SIZE} --split validation --smoke --run-id {RUN_ID}-c1 --out {OUT}/eval-smoke "
       f"--dtype {DTYPE} --kev-src {KEV_DIR} --platform kaggle", cwd=FT)
    print("C1 PASSED")

In [ ]:
# ---- c2: pilot. 300 records, one epoch; project the full run (FT-C-13) ----
import math, statistics
if "c2" in STAGES:
    PL = f"{OUT}/{RUN_ID}-c2"
    sh(f"{PY} kev/train_kev.py --size {SIZE} --data-dir data/kev --out {PL} --limit 300", cwd=FT)
    rep = json.load(open(f"{PL}/train_report.json")); tm = rep["training_metrics"]
    steps = tm["step_seconds"]; per_step = statistics.mean(steps[2:] or steps)
    n_train = 1899; full_steps = math.ceil(n_train / 8)
    est_train = per_step * full_steps
    est_total_h = (est_train + 120 + 180 + 300) / 3600          # + calibration (~210 rows), development scoring, model loads (rough)
    print(f"step {per_step:.1f}s x {full_steps} steps = {est_train/60:.0f} min training | peak GPU {tm['peak_device_bytes']/1e9:.1f} GB "
          f"| projected C2+C3 about {est_total_h:.2f} h of a {BOX_HOURS} h box")
    assert est_total_h <= BOX_HOURS, "BLOCKED: projection exceeds the time box: stop and ask the owner (FT-C-13)"
    print("C2 PASSED (projection inside the box)")

In [ ]:
# ---- c3: full run (1 epoch), then calibration on calibration.jsonl ----
if "c3" in STAGES:
    assert not os.path.exists(RUN), f"{RUN} exists: a new run id is needed (-b only with the owner's go-ahead)"
    sh(f"{PY} kev/train_kev.py --size {SIZE} --data-dir data/kev --out {RUN}", cwd=FT)
    sh(f"{PY} kev/calibrate_kev.py --run {RUN}/checkpoint --calibration data/kev/calibration.jsonl --out {RUN}/calibration", cwd=FT)
    print(open(f"{RUN}/calibration/temperature.json").read())

In [ ]:
# ---- c4_validation: round-1 runner on the validation split, report, sanity check (FT-C-12) ----
def eval_split(split):
    sh(f"{PY} kev/run_ft.py --checkpoint {RUN}/checkpoint --size {SIZE} --split {split} --run-id {RUN_ID} --out {OUT} "
       f"--dtype {DTYPE} --kev-src {KEV_DIR} --platform kaggle", cwd=FT)      # exit != 0 on any error row or validator problem
    sh(f"{PY} kev/make_report.py {OUT}/{RUN_ID}", cwd=BASE)

if "c4_validation" in STAGES:
    eval_split("validation")
    m = json.load(open(f"{OUT}/{RUN_ID}/metrics_validation.json"))
    f1 = m["route"]["macro_f1"]
    print(f"development route macro-F1 {f1:.3f} (accuracy {m['route']['accuracy']:.3f}) vs zero-shot {SANITY_MACRO_F1}")
    assert f1 > SANITY_MACRO_F1, "BLOCKED: sanity check failed (FT-C-12): stop and report"
    print("SANITY CHECK PASSED; review the validation report, then set FROZEN = True for c4_test")

In [ ]:
# ---- c4_test: once, after FROZEN ----
if "c4_test" in STAGES:
    assert FROZEN, "c4_test runs only after the configuration is frozen (FT-C-14): set FROZEN = True by hand"
    assert not os.path.exists(f"{OUT}/{RUN_ID}/test.jsonl"), "test already run for this configuration: a failed run is re-run identically, never changed"
    eval_split("test")

In [ ]:
# ---- checkpoint hashes and listing (record in run.json; upload the checkpoint folder as a private Dataset) ----
if os.path.exists(f"{RUN}/checkpoint"):
    sh(f"cd {RUN}/checkpoint && ls -la && sha256sum adapter_model*.safetensors head.pt")
sh(f"du -sh /kaggle/working; ls {OUT}")